In [ ]:
import sys

sys.path.append("..")

from datetime import datetime
from glob import glob

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from nnspike.constants import RELATIVE_POSITION_SCALE, ROI_CNN
from nnspike.data import ClassificationDataset, balance_dataset
from nnspike.models import SimpleNetClassification25
from notebooks.utils import view_data_distribution

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_course='right'
date_label = datetime.today().strftime('%m%d')

print(f"Training Course: {train_course}; Date Label: {date_label}")

## Loading Dataset

In [ ]:
label_paths = glob("../storage/labels/*.csv")
# label_paths = [path for path in label_paths
#                  if os.path.basename(path)[:8] < "20250801"]

df = pd.DataFrame()
for label_path in label_paths:

    label_df = pd.read_csv(label_path)
    df = pd.concat([df, label_df])

# Filter out the unused samples
df = df[df["use"] == True]

# Drop the unlabeled rows
df = df.dropna(subset=["target_x", "mode"])

df["mode"] = df["mode"].astype(int)
df = df[(df["mode"] == 0) | (df["mode"] == 1)]

print(f"Total number of training records: {len(df)}")
print(f"Unique behavior mode: {df['mode'].unique()}")

## Data Balancing

In [ ]:
df = df[df['motor_b_relative_position'] < 30000]
df = balance_dataset(df, 'mode', 35000, 30)
df = balance_dataset(df, 'motor_b_relative_position', 2800, 30)
view_data_distribution(df, ['mode', 'motor_b_relative_position', 'course'], [5, 30, 5])
print(f"The total number of samples is {len(df)}")

## Dataset Preparation

In [ ]:
image_paths = df['image_path'].to_list()

combined_positions = abs(df['motor_a_relative_position']) + abs(df['motor_b_relative_position'])
relative_positions = (combined_positions/RELATIVE_POSITION_SCALE).to_list()
courses = df['course'].to_list()
modes = df['mode'].to_list()

X_all = [[x, y, z] for x, y, z in zip(image_paths, relative_positions, courses)]
y_all = modes

In [ ]:
# Data Augumentation
transform = A.ShiftScaleRotate(
    shift_limit=[-0.0625, 0.0625],
    scale_limit=[-0.05, 0.05],
    rotate_limit=[-0.05, 0.05],
    border_mode=cv2.BORDER_REFLECT,
    p=0.8,
)

X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=6)

train_set = ClassificationDataset(inputs=X_train, outputs=y_train, roi=ROI_CNN, train_course=train_course, transform=transform)
val_set = ClassificationDataset(inputs=X_val, outputs=y_val, roi=ROI_CNN, train_course=train_course, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=64, shuffle=True)

## Loss Function, Optimizer, Model Initialization

In [ ]:
model_label = "simple"

criterion = nn.CrossEntropyLoss()
model = SimpleNetClassification25(num_classes=2)
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Initialize TensorBoard writer
writer = SummaryWriter()

# Training loop
num_epochs = 2
best_val_loss = float('inf')
best_model_state = None
best_epoch = 0

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_mode_loss = 0.0
    train_control_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        optimizer.zero_grad()
        inputs = [input_tensor.to(device) for input_tensor in inputs]
        labels = labels.to(device)  # Shape: (batch_size, 1) or (batch_size,)
        outputs = model(inputs[0], inputs[1])
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Calculate average losses
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [input_tensor.to(device) for input_tensor in inputs]
            labels = labels.to(device)
            outputs = model(inputs[0], inputs[1])
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    # Calculate average validation losses
    avg_val_loss = val_loss / len(val_loader)
    
    # Check if this is the best model so far
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()  # Deep copy of model state
        best_epoch = epoch + 1
        torch.save(model.state_dict(), f"../storage/models/{model_label}_cls_{train_course}_{date_label}.pt")
        print(f'  New best model found at epoch {best_epoch}!')
    
    # Log to TensorBoard
    writer.add_scalar('Loss/train_total', avg_train_loss, epoch)
    writer.add_scalar('Loss/val_total', avg_val_loss, epoch)
    writer.add_scalar('Loss/best_val', best_val_loss, epoch)
    
    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'  Train - Total: {avg_train_loss:.5f}')
    print(f'  Val   - Total: {avg_val_loss:.5f}')

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'\nLoaded best model from epoch {best_epoch} with validation loss: {best_val_loss:.5f}')
else:
    print('\nNo best model found, keeping final model state')

# Close TensorBoard writer
writer.close()


print("\nTraining completed! Feature maps have been logged to TensorBoard.")
print("To view the log, run: tensorboard --logdir=runs")

## Export to ONNX Model

In [ ]:
model.eval()

onnx_path = f"../storage/models/{model_label}_cls_{train_course}_{date_label}.onnx"

# Create dummy inputs - adjust dimensions as needed
dummy_image = torch.randn(1, 3, 66, 200)
dummy_relative_position = torch.randn(1, 1)

# Export to ONNX
torch.onnx.export(
    model,
    (dummy_image, dummy_relative_position),
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=["image", "relative_position"],
    output_names=["mode"],
)

## Example Usage

In [ ]:
import onnxruntime as ort
import numpy as np

# Load the ONNX model
onnx_path = f"../storage/models/{model_label}_cls_{train_course}_{date_label}.onnx"
session = ort.InferenceSession(onnx_path)

# Create dummy inputs (same as during export)
dummy_image = np.random.randn(1, 3, 66, 200).astype(np.float32)
dummy_relative_position = np.random.randn(1, 1).astype(np.float32)

# Prepare inputs dictionary
inputs = {
    "image": dummy_image,
    "relative_position": dummy_relative_position
}

# Run inference
outputs = session.run(["mode"], inputs)

# Get results
logits = outputs[0]  # shape: [batch_size, num_classes]
probabilities = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)  # softmax

# Get predicted classes and their confidence scores
predicted_classes = np.argmax(probabilities, axis=1)
confidence_scores = np.max(probabilities, axis=1)

print(f"Predicted classes: {predicted_classes}")
print(f"Confidence scores: {confidence_scores}")